# Executable Sakurai: Dynamics of Coherent States
**Bridging rigorous quantum formalism with interactive Python simulations.**

Welcome to *Executable Sakurai*. In this notebook, we explore the properties and time-evolution of the **Coherent State** $|\alpha\rangle$ under the Quantum Harmonic Oscillator (QHO) Hamiltonian.

In standard quantum mechanics, wavepackets typically disperse over time due to quantum interference. However, the coherent state—often dubbed the "most classical" of all quantum states—maintains its minimum-uncertainty wavepacket shape, oscillating smoothly within a quadratic potential perfectly in sync with classical equations of motion. 

Below, we digitize my handwritten derivations based on J.J. Sakurai's *Modern Quantum Mechanics*, followed by an interactive numerical simulation using `QuTiP`.

## 1. The Raw Derivation
Before translating this into executable code, it is crucial to understand the foundational operator algebra. Below is my original handwritten manuscript deriving the Fock space expansion, uncertainty principles, and time-evolution of the coherent state.
<img src="Handwritten img/Coherent_1.png" width="800">


## 2. Mathematical Formalism: Expansion in Fock Basis

We define the coherent state $|\alpha\rangle$ as the right eigenket of the non-Hermitian annihilation operator $\hat{a}$:
$$\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$$
where $\alpha$ is a complex number. 

To represent this state computationally, we must expand it in the basis of energy eigenstates (Fock states) $|n\rangle$. Assuming an expansion $|\alpha\rangle = \sum c_n |n\rangle$, we apply the annihilation operator:
$$\sum c_n \sqrt{n} |n-1\rangle = \alpha \sum c_n |n\rangle$$

By matching the coefficients, we obtain the recursive relation $c_n = c_0 \frac{\alpha^n}{\sqrt{n!}}$. Normalizing the state ($\langle\alpha|\alpha\rangle = 1$) yields $c_0 = \exp(-|\alpha|^2/2)$. Thus, the rigorous expansion is:

$$|\alpha\rangle = e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} |n\rangle$$

> **Computational Insight:** In Python (`QuTiP`), a coherent state is generated by truncating this infinite sum to a finite Hilbert space dimension $N$. The expected photon number is $\langle \alpha | \hat{N} | \alpha \rangle = |\alpha|^2$. We must choose $N \gg |\alpha|^2$ to avoid truncation errors in our simulation.

In [ ]:
# Import QuTiP and numerical libraries
import numpy as np
from qutip import destroy, coherent

# 1. System Parameters
N = 20           # Truncated Hilbert space dimension (Fock states 0 to 19)
alpha = 2.0 + 0j # Complex eigenvalue for the coherent state

# 2. Operators and State Vectors
a = destroy(N)          
psi = coherent(N, alpha) 

# 3. The Operator Shift vs. Scalar Multiplication
a_psi = a * psi 
alpha_psi = alpha * psi

print("--- Verifying the Eigenvalue Equation: a|α> = α|α> ---")
print(f"First 5 amplitudes of |α>:            {np.round(psi.full()[:5].flatten(), 4)}")
print(f"First 5 amplitudes of a|α> (shifted): {np.round(a_psi.full()[:5].flatten(), 4)}")
print(f"First 5 amplitudes of α * |α>:        {np.round(alpha_psi.full()[:5].flatten(), 4)}\n")

print("--- Computational Verification ---")
is_exact_match = np.allclose(a_psi.full(), alpha_psi.full())
print(f"Do the vectors match exactly across all N={N} dimensions? {is_exact_match}")

--- Verifying the Eigenvalue Equation: a|α> = α|α> ---
First 5 amplitudes of |α>:            [0.1353+0.j 0.2707+0.j 0.3828+0.j 0.442 +0.j 0.442 +0.j]
First 5 amplitudes of a|α> (shifted): [0.2707+0.j 0.5413+0.j 0.7656+0.j 0.884 +0.j 0.884 +0.j]
First 5 amplitudes of α * |α>:        [0.2707+0.j 0.5413+0.j 0.7656+0.j 0.884 +0.j 0.884 +0.j]

--- Computational Verification ---
Do the vectors match exactly across all N=20 dimensions? False


### The Truncation Trap: Why did it return `False`?

On a theoretical chalkboard, the Hilbert space is infinite-dimensional, ensuring the fundamental commutation relation $[\hat{a}, \hat{a}^\dagger] = 1$ holds perfectly. But why does a strict numerical verification in our code return `False`?

This is not a bug in our code; rather, we have run into a classic pitfall in computational quantum physics: the **Truncation Trap**.

1. **Boundary Breakdown:** In computational simulations, we must truncate the infinite Hilbert space to a finite dimension (here, $N=20$). When the annihilation operator $\hat{a}$ acts on the highest boundary state $|N-1\rangle$, it attempts to pull amplitude from a non-existent $|N\rangle$ state. This artificial cutoff explicitly breaks the commutation relation at the boundary.
2. **Black-box Algorithmic Error:** By default, `QuTiP` does *not* directly plug into our derived $c_n$ coefficient formula! Instead, it generates the coherent state using the matrix exponential of the **Displacement Operator**: $|\alpha\rangle = \exp(\alpha \hat{a}^\dagger - \alpha^* \hat{a})|0\rangle$. Because the commutation relation is violated in finite-dimensional matrices, the continuous-space Baker-Campbell-Hausdorff (BCH) expansion is no longer strictly valid. This introduces a minuscule, yet mathematically fatal, truncation error.

To fix this, we must "teach" the computer to think like a theoretical physicist: we will abandon the default matrix exponential, force the use of our hand-derived **Analytical Formula**, and safely evaluate the state within the physical subspace (ignoring the artificial boundary).

In [ ]:
print("--- Resolving the Truncation Trap ---")

psi_analytic = coherent(N, alpha, method='analytic')

# 1. Apply operators again
a_psi_ana = a * psi_analytic
alpha_psi_ana = alpha * psi_analytic

# 2. Verify in the "Safe Physical Subspace"
# We slice the arrays [:-2] to completely ignore the top 2 artificial boundary states 
safe_match_analytic = np.allclose(a_psi_ana.full()[:-2], alpha_psi_ana.full()[:-2])

print(f"Do they match in the safe physical subspace using the analytical method? {safe_match_analytic}")
print("\nTheoretical victory! The mathematics holds perfectly when computational boundaries are respected.")

--- Resolving the Truncation Trap ---
Do they match in the safe physical subspace using the analytical method? True

Theoretical victory! The mathematics holds perfectly when computational boundaries are respected.


## 3. The Minimum Uncertainty State

Why is the coherent state considered "classical"? We look at the variances in position $\hat{x}$ and momentum $\hat{p}$.
Using the creation and annihilation operators:
$$\hat{x} = \sqrt{\frac{\hbar}{2m\omega}} (\hat{a}^\dagger + \hat{a}), \quad \hat{p} = i\sqrt{\frac{m\hbar\omega}{2}} (\hat{a}^\dagger - \hat{a})$$

Evaluating the expectation values $\langle \hat{x}^2 \rangle - \langle \hat{x} \rangle^2$, we find:
$$\Delta x = \sqrt{\frac{\hbar}{2m\omega}}, \quad \Delta p = \sqrt{\frac{m\hbar\omega}{2}}$$

Hence, the uncertainty product hits the absolute theoretical minimum allowed by Heisenberg's principle:
$$\Delta x \cdot \Delta p = \frac{\hbar}{2}$$

## 4. Time Evolution: The Rigid Wavepacket

The most fascinating property of the coherent state emerges when we evolve it in time under the Harmonic Oscillator Hamiltonian $H = \hbar\omega(\hat{a}^\dagger \hat{a} + \frac{1}{2})$. 

Applying the time-evolution operator $\hat{U}(t) = \exp(-i\hat{H}t/\hbar)$ to our Fock expansion:

$$|\alpha, t\rangle = e^{-i\hat{H}t/\hbar} |\alpha\rangle = e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} e^{-i\omega(\frac{1}{2} + n)t} |n\rangle$$

By factoring out the zero-point energy phase $e^{-i\omega t/2}$, we can absorb the dynamic phase into the eigenvalue $\alpha$:

$$|\alpha, t\rangle = e^{-i\omega t / 2} \left[ e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{(\alpha e^{-i\omega t})^n}{\sqrt{n!}} |n\rangle \right]$$

$$|\alpha, t\rangle = e^{-i\omega t / 2} \left| \alpha e^{-i\omega t} \right\rangle$$

> **Physical Insight (The Core of the Simulation):** > The state *remains a coherent state* at all times! The complex eigenvalue $\alpha$ simply rotates in the complex plane: $\alpha(t) = \alpha(0) e^{-i\omega t}$. 
> In phase space (the Wigner function), this corresponds to a rigid Gaussian wavepacket moving in a perfect circle, exactly mimicking a classical pendulum, without spreading!

### Interactive Exploration: The Phase Space Showdown

Numbers alone do not capture the profound difference between these states. Let's visualize it interactively. 

Using `ipywidgets` and `QuTiP`'s Wigner function plotting, you can now explore the phase space yourself. 
* Use the **Fock |n>** slider to increase the energy level of the energy eigenstate. Watch how the wavepacket forms complex quantum interference rings and how its uncertainty strictly grows.
* Use the **Coherent α** slider to displace the coherent state. Notice how it rigidly maintains its shape and its minimum uncertainty product of `0.5000`, regardless of where it is in phase space.

*Run the cell below and interact with the sliders!*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import destroy, coherent, fock, variance, plot_wigner
import ipywidgets as widgets
from ipywidgets import interact

def interactive_quantum_contrast(n_level, alpha_val):
    # Use a slightly larger Hilbert space to avoid truncation for larger n and alpha
    N = 35 
    a = destroy(N)
    
    # Define Position (x) and Momentum (p) operators
    x = (a.dag() + a) / np.sqrt(2)
    p = 1j * (a.dag() - a) / np.sqrt(2)
    
    # --- 1. Compute Coherent State ---
    psi_coh = coherent(N, alpha_val, method='analytic')
    var_x_coh = variance(x, psi_coh)
    var_p_coh = variance(p, psi_coh)
    uncert_coh = np.sqrt(var_x_coh) * np.sqrt(var_p_coh)
    
    # --- 2. Compute Fock State ---
    psi_fock = fock(N, n_level)
    var_x_fock = variance(x, psi_fock)
    var_p_fock = variance(p, psi_fock)
    uncert_fock = np.sqrt(var_x_fock) * np.sqrt(var_p_fock)
    
    # --- 3. Dynamic Plotting (Phase Space Wigner Functions) ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot Fock State
    plot_wigner(psi_fock, fig=fig, ax=axes[0], cmap='RdBu', alpha_max=5)
    axes[0].set_title(f"Fock State |n={n_level}>\nUncertainty Product = {uncert_fock:.4f}", fontsize=14)
    axes[0].set_xlabel("Position x")
    axes[0].set_ylabel("Momentum p")
    
    # Plot Coherent State
    plot_wigner(psi_coh, fig=fig, ax=axes[1], cmap='RdBu', alpha_max=5)
    axes[1].set_title(f"Coherent State |α={alpha_val}>\nUncertainty Product = {uncert_coh:.4f}", fontsize=14)
    axes[1].set_xlabel("Position x")
    
    plt.tight_layout()
    plt.show()
    
    # --- 4. Dynamic Text Output ---
    print("-" * 65)
    print(f"Absolute Theoretical Minimum (Heisenberg Limit): 0.5000")
    print(f"Current Coherent State |α={alpha_val}> Product:      {uncert_coh:.4f}")
    print(f"Current Fock State |n={n_level}> Product:           {uncert_fock:.4f}")
    print("-" * 65)
    
    if uncert_fock > uncert_coh:
        print("\nInsight: As n increases, the Fock state spreads vastly in phase space (quantum fuzziness).")
        print("Conversely, the coherent state remains a perfectly localized classical-like particle!")

# --- 5. Generate the Interactive UI ---
interact(interactive_quantum_contrast, 
         n_level=widgets.IntSlider(min=0, max=12, step=1, value=5, description='Fock |n>:', continuous_update=False),
         alpha_val=widgets.FloatSlider(min=0.0, max=4.0, step=0.5, value=2.0, description='Coherent α:', continuous_update=False));


## 5. Interactive Simulation: QuTiP Implementation

We have proven the theory. Now, let's build the numerical simulation. 
Below, we will use `QuTiP` to:
1. Construct the Hamiltonian and the initial coherent state.
2. Evolve the system using the Schrödinger equation solver (`sesolve`).
3. Plot the Wigner function to visualize the phase-space rotation.